# DG baseline · SegFormer-B2 · Cityscapes only

Train Cityscapes 2.975 ảnh, chọn best bằng Cityscapes val 500, 20 epoch. **Không thêm ACDC; không train hoặc chọn best bằng ảnh/nhãn ACDC.** Đây là baseline, chưa có module DG cải tiến.

Input: code ZIP DG mới + Cityscapes train/val. Fresh: RESUME_INPUT=None. Resume: dùng kết quả DG của đúng lượt, cùng mount. Lưu ZIP sau mỗi epoch hoàn thành; epoch đang dở chạy lại. Không nạp ZIP supervised cũ. Chọn Save Version → Save & Run All.

In [ ]:
"""Standard-library source bootstrap shared by the three model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_research_campaign.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','requirements-demo.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os,sys,json,hashlib,time
STARTED=time.monotonic()
CODE_INPUT=Path('/kaggle/input')
CITYSCAPES_ROOT=None
RESUME_INPUT=None  # DG result ZIP or extracted DG workspace, never supervised results
WORK=Path('/kaggle/working/dg_b2_baseline')
REPO=Path('/kaggle/working/project_dg')
ARCHIVE=Path('/kaggle/working/dg_b2_baseline_seed42.zip')
assert not WORK.exists() and not REPO.exists(),'Fresh session required; preserve previous result first'
if RESUME_INPUT is None:
    copy_source(resolve_source(CODE_INPUT,Path('/kaggle/input')),REPO)
else:
    previous=Path(RESUME_INPUT)
    if previous.is_file():extract_safe(previous,WORK)
    else:
        candidates=[previous] if (previous/'dg_plan.json').is_file() else [p.parent for p in previous.rglob('dg_plan.json')]
        assert len(candidates)==1,f'Set RESUME_INPUT to one DG workspace: {candidates}'
        shutil.copytree(candidates[0],WORK)
    plan=json.loads((WORK/'dg_plan.json').read_text())
    assert plan['experiment_protocol']=='cityscapes_only_dg_v1','Reject supervised or mismatched results'
    copy_source(WORK/'source.zip' if (WORK/'source.zip').is_file() else WORK/'source',REPO)
    assert all(hashlib.sha256((REPO/n).read_bytes()).hexdigest()==h for n,h in plan['source_hashes'].items())
    if not (WORK/'source.zip').is_file():
        with zipfile.ZipFile(WORK/'source.zip','w',zipfile.ZIP_DEFLATED) as z:
            for name in plan['source_hashes']:z.write(REPO/name,name)
            for name in ('requirements.txt','THIRD_PARTY_NOTICES.md','third_party/transformers/LICENSE'):
                if (REPO/name).is_file():z.write(REPO/name,name)
    CITYSCAPES_ROOT=Path(plan['city_root'])
assert (REPO/'scripts/run_dg.py').is_file(),'Upload the new DG project-code.zip'
if CITYSCAPES_ROOT is None:
    candidates=sorted({p.parent for p in Path('/kaggle/input').rglob('leftImg8bit') if p.is_dir()
                       and all((p.parent/n).is_dir() for n in ['leftImg8bit/train','gtFine/train','leftImg8bit/val','gtFine/val'])})
    assert len(candidates)==1,f'Set CITYSCAPES_ROOT explicitly: {candidates}'
    CITYSCAPES_ROOT=candidates[0]
os.chdir(REPO);sys.path.insert(0,str(REPO))
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
print('DG baseline: Cityscapes only:',CITYSCAPES_ROOT)


In [ ]:
import subprocess
def run(*args):subprocess.run([sys.executable,'-u',*map(str,args)],check=True,cwd=REPO)
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow>=10,<12',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torch
assert torch.cuda.is_available(),'Enable GPU'
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2]))>=(2,6),'Require Torch>=2.6'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
from src.experiments.packaging import export_workspace
from IPython.display import FileLink,display
remaining=10-(time.monotonic()-STARTED)/3600
try:
    if remaining>0:
        run('-m','scripts.run_dg','--city',CITYSCAPES_ROOT,'--root',WORK,'--archive',ARCHIVE,
            '--session-hours',remaining)
finally:
    if WORK.exists():export_workspace(WORK,ARCHIVE)
    if ARCHIVE.is_file():print('Download:',ARCHIVE);display(FileLink(str(ARCHIVE)))


Hoàn thành: giữ dg_plan.json, frozen_candidate.json và run/best. Chỉ đánh giá ACDC sau khi khóa phương pháp/checkpoint. Điểm ACDC đã xem trong lịch sử không phải test độc lập. Chưa có kết quả GPU cho protocol mới này.